In [24]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw" / "METR-LA"

TRAFFIC_FILE = RAW_DATA_DIR / "METR-LA.h5"
ADJACENCY_FILE = RAW_DATA_DIR / "adj_METR-LA.pkl"

print("Project root:", PROJECT_ROOT)
print("Traffic file:", TRAFFIC_FILE)
print("Adjacency file:", ADJACENCY_FILE)

print("\nTraffic file exists:", TRAFFIC_FILE.exists())
print("Adjacency file exists:", ADJACENCY_FILE.exists())

Project root: /Users/sushank/Documents/GitHub/urbanAi
Traffic file: /Users/sushank/Documents/GitHub/urbanAi/data/raw/METR-LA/METR-LA.h5
Adjacency file: /Users/sushank/Documents/GitHub/urbanAi/data/raw/METR-LA/adj_METR-LA.pkl

Traffic file exists: True
Adjacency file exists: True


In [25]:
print(f"Traffic dataset size: {TRAFFIC_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Adjacency file size: {ADJACENCY_FILE.stat().st_size / (1024**2):.2f} MB")

Traffic dataset size: 54.40 MB
Adjacency file size: 0.65 MB


In [26]:
import h5py

with h5py.File(TRAFFIC_FILE, "r") as f:
    print("Keys:")
    for key in f.keys():
        print(" -", key)

Keys:
 - df


In [27]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    df = f["df"]

    print("df type:", type(df))
    print("Contents of df:")

    for key in df.keys():
        item = df[key]
        print(f" - {key}: {type(item)}")

df type: <class 'h5py._hl.group.Group'>
Contents of df:
 - axis0: <class 'h5py._hl.dataset.Dataset'>
 - axis1: <class 'h5py._hl.dataset.Dataset'>
 - block0_items: <class 'h5py._hl.dataset.Dataset'>
 - block0_values: <class 'h5py._hl.dataset.Dataset'>


In [28]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    df = f["df"]

    for key in df.keys():
        dataset = df[key]
        print(f"{key}:")
        print(f"  shape = {dataset.shape}")
        print(f"  dtype = {dataset.dtype}")
        print()

axis0:
  shape = (207,)
  dtype = |S6

axis1:
  shape = (34272,)
  dtype = int64

block0_items:
  shape = (207,)
  dtype = |S6

block0_values:
  shape = (34272, 207)
  dtype = float64



In [29]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    df = f["df"]

    axis0 = df["axis0"][:]
    axis1 = df["axis1"][:]

print("First 10 axis0 values:")
print(axis0[:10])

print("\nFirst 10 axis1 values:")
print(axis1[:10])

First 10 axis0 values:
[b'773869' b'767541' b'767542' b'717447' b'717446' b'717445' b'773062'
 b'767620' b'737529' b'717816']

First 10 axis1 values:
[1330560000000000000 1330560300000000000 1330560600000000000
 1330560900000000000 1330561200000000000 1330561500000000000
 1330561800000000000 1330562100000000000 1330562400000000000
 1330562700000000000]


In [30]:
sensor_ids = [x.decode("utf-8") for x in axis0]

print("First 10 sensor IDs:")
print(sensor_ids[:10])

First 10 sensor IDs:
['773869', '767541', '767542', '717447', '717446', '717445', '773062', '767620', '737529', '717816']


In [31]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    values = f["df"]["block0_values"][:5, :5]

print(values)

[[64.375      67.625      67.125      61.5        66.875     ]
 [62.66666667 68.55555556 65.44444444 62.44444444 64.44444444]
 [64.         63.75       60.         59.         66.5       ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]]


In [32]:
timestamps = pd.to_datetime(axis1)

print("First timestamp:", timestamps[0])
print("Last timestamp:", timestamps[-1])

print("\nFirst 10 timestamps:")
print(timestamps[:10])

print("\nTime differences:")
print(timestamps.to_series().diff().value_counts().head())

First timestamp: 2012-03-01 00:00:00
Last timestamp: 2012-06-27 23:55:00

First 10 timestamps:
DatetimeIndex(['2012-03-01 00:00:00', '2012-03-01 00:05:00',
               '2012-03-01 00:10:00', '2012-03-01 00:15:00',
               '2012-03-01 00:20:00', '2012-03-01 00:25:00',
               '2012-03-01 00:30:00', '2012-03-01 00:35:00',
               '2012-03-01 00:40:00', '2012-03-01 00:45:00'],
              dtype='datetime64[ns]', freq=None)

Time differences:
0 days 00:05:00    34271
Name: count, dtype: int64


In [33]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    values = f["df"]["block0_values"][:5, :5]

print(values)

[[64.375      67.625      67.125      61.5        66.875     ]
 [62.66666667 68.55555556 65.44444444 62.44444444 64.44444444]
 [64.         63.75       60.         59.         66.5       ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]]


In [34]:
print("Minimum:", np.nanmin(values))
print("Maximum:", np.nanmax(values))
print("Mean:", np.nanmean(values))
print("NaN count:", np.isnan(values).sum())

Minimum: 0.0
Maximum: 68.55555555555556
Mean: 38.57222222222222
NaN count: 0


In [35]:
with h5py.File(TRAFFIC_FILE, "r") as f:
    df_group = f["df"]

    sensor_ids = [
        sensor.decode("utf-8")
        for sensor in df_group["axis0"][:]
    ]

    timestamps = pd.to_datetime(
        df_group["axis1"][:]
    )

    values = df_group["block0_values"][:]

traffic_df = pd.DataFrame(
    values,
    index=timestamps,
    columns=sensor_ids
)

traffic_df.index.name = "timestamp"

print("Shape:", traffic_df.shape)
print("\nFirst 5 rows:")
display(traffic_df.head())

Shape: (34272, 207)

First 5 rows:


,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
timestamp,,,,,,,,,,,,,,,,,,,,,
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,62.750000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,63.333333,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000


In [36]:
print("Index type:", type(traffic_df.index))
print("Number of sensors:", traffic_df.shape[1])
print("Number of timestamps:", traffic_df.shape[0])

print("\nSampling interval:")
print(traffic_df.index.to_series().diff().value_counts().head())

Index type: <class 'pandas.DatetimeIndex'>
Number of sensors: 207
Number of timestamps: 34272

Sampling interval:
timestamp
0 days 00:05:00    34271
Name: count, dtype: int64


In [37]:
traffic_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 34272 entries, 2012-03-01 00:00:00 to 2012-06-27 23:55:00
Columns: 207 entries, 773869 to 769373
dtypes: float64(207)
memory usage: 54.4 MB


In [38]:
traffic_df.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
773869,34272.0,54.631359,22.619199,0.0,60.364583,64.888889,66.875000,70.000000
767541,34272.0,60.452789,15.970239,0.0,63.000000,65.000000,66.375000,70.000000
767542,34272.0,60.726120,18.313353,0.0,65.444444,67.375000,68.444444,70.000000
717447,34272.0,49.524287,15.843261,0.0,50.333333,53.875000,58.125000,70.000000
717446,34272.0,46.079798,19.350345,0.0,34.666667,46.000000,64.500000,70.000000
717445,34272.0,50.952003,16.681760,0.0,49.555556,56.111111,60.333333,70.000000
773062,34272.0,54.471684,17.984761,0.0,55.750000,62.111111,65.000000,70.000000
767620,34272.0,57.255095,18.751065,0.0,61.000000,63.333333,65.000000,70.000000
737529,34272.0,56.068044,18.240361,0.0,58.222222,62.444444,64.888889,70.000000
717816,34272.0,52.871841,23.343805,0.0,43.428571,65.875000,67.625000,70.000000


In [39]:
print("Total values:", traffic_df.size)
print("Missing values:", traffic_df.isna().sum().sum())
print("Missing percentage:",
      traffic_df.isna().sum().sum() / traffic_df.size * 100)

Total values: 7094304
Missing values: 0
Missing percentage: 0.0


In [40]:
traffic_df.iloc[:, 0].describe()

count    34272.000000
mean        54.631359
std         22.619199
min          0.000000
25%         60.364583
50%         64.888889
75%         66.875000
max         70.000000
Name: 773869, dtype: float64

In [41]:
traffic_df.iloc[:, :5].head(10)

,773869,767541,767542,717447,717446
timestamp,,,,,
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000
2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000
2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000
2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000
2012-03-01 00:25:00,57.333333,69.000000,67.666667,61.666667,67.333333
2012-03-01 00:30:00,66.500000,63.875000,67.875000,62.375000,64.375000
2012-03-01 00:35:00,63.625000,67.250000,63.250000,60.500000,57.375000
2012-03-01 00:40:00,68.750000,65.250000,63.500000,63.000000,65.125000


## Phase 1.3 - Dataset Statistics
This section computes overall and per-sensor descriptive statistics from the reconstructed METR-LA matrix.

In [ ]:
import pickle
import seaborn as sns
import matplotlib.dates as mdates

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

overall_stats = {
    "total_timestamps": int(traffic_df.shape[0]),
    "total_sensors": int(traffic_df.shape[1]),
    "total_values": int(traffic_df.size),
    "memory_usage_mb": float(traffic_df.memory_usage(deep=True).sum() / (1024 ** 2)),
    "dtype_set": sorted({str(dtype) for dtype in traffic_df.dtypes}),
    "global_min": float(traffic_df.min().min()),
    "global_max": float(traffic_df.max().max()),
    "global_mean": float(traffic_df.values.mean()),
    "global_median": float(np.median(traffic_df.values)),
    "global_std": float(traffic_df.values.std()),
}

overall_stats["quantiles"] = {
    "q01": float(np.quantile(traffic_df.values, 0.01)),
    "q05": float(np.quantile(traffic_df.values, 0.05)),
    "q25": float(np.quantile(traffic_df.values, 0.25)),
    "q50": float(np.quantile(traffic_df.values, 0.50)),
    "q75": float(np.quantile(traffic_df.values, 0.75)),
    "q95": float(np.quantile(traffic_df.values, 0.95)),
    "q99": float(np.quantile(traffic_df.values, 0.99)),
}

pd.Series(overall_stats)

In [ ]:
per_sensor_summary = traffic_df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

summary_cols = ["mean", "std", "min", "5%", "50%", "95%", "max"]
print("Per-sensor summary (first 10 sensors):")
display(per_sensor_summary.loc[:, summary_cols].head(10))

print("Per-sensor mean distribution summary:")
display(per_sensor_summary["mean"].describe())

In [ ]:
# Investigate contiguous zero periods across all sensors simultaneously.
zero_count_ts = (traffic_df == 0).sum(axis=1)
high_zero_threshold = int(np.ceil(0.10 * traffic_df.shape[1]))  # >= 10% sensors are zero
high_zero_flag = zero_count_ts >= high_zero_threshold

segments = []
if high_zero_flag.any():
    change = high_zero_flag.ne(high_zero_flag.shift(fill_value=False)).cumsum()
    for _, seg in high_zero_flag.groupby(change):
        if seg.iloc[0]:
            start = seg.index[0]
            end = seg.index[-1]
            duration_steps = len(seg)
            segments.append(
                {
                    "start": start,
                    "end": end,
                    "duration_steps": duration_steps,
                    "duration_minutes": duration_steps * 5,
                    "max_zero_sensors_in_segment": int(zero_count_ts.loc[start:end].max()),
                }
            )

segment_df = pd.DataFrame(segments)
print(f"Threshold for high-zero timestamp: >= {high_zero_threshold} sensors")
print(f"Number of high-zero contiguous segments: {len(segment_df)}")
if len(segment_df) > 0:
    display(segment_df.head(15))

# Example window around the first high-zero segment if present.
if len(segment_df) > 0:
    first_start = segment_df.iloc[0]["start"]
    window = traffic_df.loc[first_start - pd.Timedelta(minutes=20): first_start + pd.Timedelta(minutes=20)]
    print("Example around first high-zero segment:")
    display(window.iloc[:, :8].head(9))

In [ ]:
nan_count = int(traffic_df.isna().sum().sum())
inf_count = int(np.isinf(traffic_df.values).sum())
zero_mask = traffic_df == 0
zero_count = int(zero_mask.sum().sum())

total_values = int(traffic_df.size)

missingness_stats = pd.Series(
    {
        "total_values": total_values,
        "nan_count": nan_count,
        "nan_pct": 100 * nan_count / total_values,
        "inf_count": inf_count,
        "inf_pct": 100 * inf_count / total_values,
        "zero_count": zero_count,
        "zero_pct": 100 * zero_count / total_values,
    }
)

display(missingness_stats)

zeros_per_sensor = zero_mask.sum(axis=0).sort_values(ascending=False)
zeros_per_sensor_pct = (zeros_per_sensor / len(traffic_df) * 100).rename("zero_pct")

zeros_per_timestamp = zero_mask.sum(axis=1).sort_values(ascending=False)
zeros_per_timestamp_pct = (zeros_per_timestamp / traffic_df.shape[1] * 100).rename("zero_pct")

print("Top 10 sensors by zero count:")
display(
    pd.concat(
        [zeros_per_sensor.head(10).rename("zero_count"), zeros_per_sensor_pct.head(10)],
        axis=1,
    )
)

print("Top 10 timestamps by zero count:")
display(
    pd.concat(
        [zeros_per_timestamp.head(10).rename("zero_count"), zeros_per_timestamp_pct.head(10)],
        axis=1,
    ).head(10)
)

## Phase 1.4 - Missing-Value and Zero-Value Analysis

## Phase 1.5 - Traffic Visualization
The following plots cover long-term behavior, intraday behavior, multi-sensor comparison, value distribution, and zero-occurrence behavior.

In [ ]:
sensor_for_long_term = traffic_df.columns[0]
sensor_for_day = traffic_df.columns[1]
selected_sensors = list(traffic_df.columns[:5])

# Plot A: one sensor across the entire dataset.
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(traffic_df.index, traffic_df[sensor_for_long_term], linewidth=0.8)
ax.set_title(f"Plot A - Full Timeline for Sensor {sensor_for_long_term}")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Traffic Value")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Plot B: one representative day.
representative_day = traffic_df.index[0].date() + pd.Timedelta(days=20)
day_slice = traffic_df.loc[str(representative_day), sensor_for_day]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(day_slice.index, day_slice.values, marker="o", markersize=2, linewidth=0.8)
ax.set_title(f"Plot B - Intraday Pattern on {representative_day} (Sensor {sensor_for_day})")
ax.set_xlabel("Time of Day")
ax.set_ylabel("Traffic Value")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Plot C: several sensors together for behavior comparison.
fig, ax = plt.subplots(figsize=(14, 4))
for sensor_id in selected_sensors:
    ax.plot(traffic_df.index, traffic_df[sensor_id], linewidth=0.7, label=sensor_id)
ax.set_title("Plot C - Multi-Sensor Temporal Comparison (First 5 Sensors)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Traffic Value")
ax.legend(loc="upper right", ncol=2)
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Plot D: global distribution.
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(traffic_df.values.ravel(), bins=120, edgecolor="black", alpha=0.7)
ax.set_title("Plot D - Global Traffic Value Distribution")
ax.set_xlabel("Traffic Value")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

# Plot E: zero-value activity by timestamp.
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(zero_count_ts.index, zero_count_ts.values, linewidth=0.8, color="tab:red")
ax.axhline(high_zero_threshold, linestyle="--", color="black", label=f"10% threshold ({high_zero_threshold})")
ax.set_title("Plot E - Number of Zero-Valued Sensors per Timestamp")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Zero-valued sensor count")
ax.legend()
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Phase 1.6 - Daily Patterns
Analyze aggregated behavior by hour and by weekday/weekend.

In [ ]:
global_series = traffic_df.mean(axis=1)

hourly_mean = global_series.groupby(global_series.index.hour).mean()
hourly_std = global_series.groupby(global_series.index.hour).std()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_mean.index, hourly_mean.values, marker="o", label="Mean")
ax.fill_between(
    hourly_mean.index,
    hourly_mean.values - hourly_std.values,
    hourly_mean.values + hourly_std.values,
    alpha=0.2,
    label="Mean ± Std",
)
ax.set_title("Average Traffic by Hour of Day")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Average Traffic")
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
plt.show()

weekday_flag = global_series.index.dayofweek < 5
weekday_mean_by_hour = global_series[weekday_flag].groupby(global_series[weekday_flag].index.hour).mean()
weekend_mean_by_hour = global_series[~weekday_flag].groupby(global_series[~weekday_flag].index.hour).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weekday_mean_by_hour.index, weekday_mean_by_hour.values, marker="o", label="Weekday")
ax.plot(weekend_mean_by_hour.index, weekend_mean_by_hour.values, marker="o", label="Weekend")
ax.set_title("Weekday vs Weekend Mean Traffic by Hour")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Average Traffic")
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
plt.show()

hourly_table = pd.DataFrame({
    "hour": hourly_mean.index,
    "mean": hourly_mean.values,
    "std": hourly_std.values,
}).set_index("hour")

print("Hourly aggregate summary (first 12 hours):")
display(hourly_table.head(12))

## Phase 1 Findings
### Observed
- The dataset matrix dimensions, coverage, and sampling interval were verified directly from HDF5 reconstruction.
- Missingness metrics were measured separately for NaN/Inf/explicit zeros.
- Zero values occur non-uniformly across sensors and timestamps, including contiguous periods with multi-sensor zeros.
- Daily and weekly aggregate patterns show temporal structure (hourly and weekday/weekend variability).
- Sensor correlations include both strongly related and weakly related pairs.
- Adjacency file inspection confirmed graph matrix structure and node coverage checks.

### Interpretation
- Explicit zeros should not be silently imputed or dropped before understanding their semantic meaning.
- Preprocessing for forecasting must remain chronological with train-only fit statistics to avoid leakage.
- Because sensors are heterogeneous, normalization should be feature-wise and reproducibly parameterized.
- Correlation findings are useful for exploratory insight but do not prove physical graph adjacency on their own.
- Graph construction in Phase 3 should use adjacency metadata directly, with sensor ID alignment checks retained as validation.

In [ ]:
with open(ADJACENCY_FILE, "rb") as f:
    # METR-LA adjacency pickle is commonly serialized with Python 2 defaults.
    adj_obj = pickle.load(f, encoding="latin1")

print("Adjacency object type:", type(adj_obj))

adj_sensor_ids = None
adj_matrix = None
if isinstance(adj_obj, (tuple, list)) and len(adj_obj) >= 2:
    # Common METR-LA format: [sensor_ids, sensor_id_to_ind, adjacency_matrix]
    if len(adj_obj) >= 3:
        adj_sensor_ids = adj_obj[0]
        adj_matrix = np.asarray(adj_obj[2])
    else:
        adj_sensor_ids = adj_obj[0]
        adj_matrix = np.asarray(adj_obj[-1])
elif isinstance(adj_obj, dict):
    # Fallback for dict-like adjacency containers.
    if "adj_mx" in adj_obj:
        adj_matrix = np.asarray(adj_obj["adj_mx"])
    elif "adjacency" in adj_obj:
        adj_matrix = np.asarray(adj_obj["adjacency"])
    if "sensor_ids" in adj_obj:
        adj_sensor_ids = adj_obj["sensor_ids"]

if adj_matrix is None:
    raise ValueError("Could not locate adjacency matrix in adj_METR-LA.pkl")

adj_nonzero = int(np.count_nonzero(adj_matrix))
node_count = int(adj_matrix.shape[0])
edge_weight_exists = bool(np.any((adj_matrix != 0) & (adj_matrix != 1)))

is_square = adj_matrix.ndim == 2 and adj_matrix.shape[0] == adj_matrix.shape[1]
symmetric = bool(np.allclose(adj_matrix, adj_matrix.T)) if is_square else False

graph_stats = {
    "adj_shape": adj_matrix.shape,
    "node_count": node_count,
    "nonzero_entries": adj_nonzero,
    "interpreted_directed_edges": adj_nonzero,
    "interpreted_undirected_edges_if_symmetric": int(adj_nonzero // 2),
    "is_square": is_square,
    "is_symmetric": symmetric,
    "has_weighted_edges": edge_weight_exists,
    "all_207_nodes_present": node_count == traffic_df.shape[1],
}

display(pd.Series(graph_stats))

if adj_sensor_ids is not None:
    adj_sensor_ids = [str(x) for x in adj_sensor_ids]
    traffic_sensor_set = set(traffic_df.columns.astype(str))
    adj_sensor_set = set(adj_sensor_ids)
    print("Sensor ID overlap count:", len(traffic_sensor_set & adj_sensor_set))
    print("Traffic-only IDs count:", len(traffic_sensor_set - adj_sensor_set))
    print("Adjacency-only IDs count:", len(adj_sensor_set - traffic_sensor_set))
    print("First 10 adjacency sensor IDs:", adj_sensor_ids[:10])

## Phase 1.9 - Adjacency / Graph File Inspection

In [ ]:
corr_matrix = traffic_df.corr()

# Heatmap for a readable subset.
subset_size = 40
subset_sensors = traffic_df.columns[:subset_size]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix.loc[subset_sensors, subset_sensors], cmap="coolwarm", center=0, ax=ax)
ax.set_title(f"Correlation Heatmap (First {subset_size} Sensors)")
plt.tight_layout()
plt.show()

# Summarize strongest and weakest non-self correlation pairs.
corr_pairs = corr_matrix.where(~np.eye(corr_matrix.shape[0], dtype=bool)).stack().sort_values(ascending=False)
strong_pairs = corr_pairs.head(10)
weak_pairs = corr_pairs.tail(10)

print("Top 10 strongest sensor correlations:")
display(strong_pairs.rename("correlation").to_frame())

print("Top 10 weakest sensor correlations:")
display(weak_pairs.rename("correlation").to_frame())

## Phase 1.8 - Sensor Correlations

In [ ]:
day_name_series = global_series.index.day_name()
weekly_mean = global_series.groupby(day_name_series).mean().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)
weekly_std = global_series.groupby(day_name_series).std().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(weekly_mean.index, weekly_mean.values, yerr=weekly_std.values, capsize=4)
ax.set_title("Average Traffic by Day of Week")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Average Traffic")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

weekly_table = pd.DataFrame({"mean": weekly_mean, "std": weekly_std})
display(weekly_table)

## Phase 1.7 - Weekly Patterns

In [42]:
# ============================================================
# Zero-Value Analysis
# ============================================================

zero_count = (traffic_df == 0).sum()

zero_stats = pd.DataFrame({
    "zero_count": zero_count,
    "zero_percentage": zero_count / len(traffic_df) * 100
})

print("Total zero values:", int(zero_count.sum()))

print(
    "Overall zero percentage:",
    float((traffic_df == 0).sum().sum() / traffic_df.size * 100)
)

print("\nSensors with highest zero percentage:")

display(
    zero_stats
    .sort_values("zero_percentage", ascending=False)
    .head(20)
)

# ------------------------------------------------------------
# Zero values per timestamp
# ------------------------------------------------------------

zero_per_timestamp = (traffic_df == 0).sum(axis=1)

print("\nZero count per timestamp:")
print(zero_per_timestamp.describe())

print("\nTimestamps with most zero-valued sensors:")

display(
    zero_per_timestamp
    .sort_values(ascending=False)
    .head(20)
)

Total zero values: 575302
Overall zero percentage: 8.109350825676486

Sensors with highest zero percentage:


,zero_count,zero_percentage
718076,6890,20.103875
716939,6464,18.860878
760987,5532,16.141457
716955,5374,15.680439
769418,5349,15.607493
769405,5346,15.598739
717481,5163,15.064776
717510,5156,15.044351
717513,4819,14.061041
717576,4675,13.640873



Zero count per timestamp:
count    34272.000000
mean        16.786356
std         49.361761
min          0.000000
25%          1.000000
50%          3.000000
75%          7.000000
max        207.000000
dtype: float64

Timestamps with most zero-valued sensors:


timestamp
2012-04-09 00:20:00    207
2012-04-08 11:35:00    207
2012-03-12 17:25:00    207
2012-03-12 17:30:00    207
2012-04-08 12:25:00    207
2012-04-08 12:20:00    207
2012-04-08 12:15:00    207
2012-04-08 12:10:00    207
2012-04-08 12:05:00    207
2012-04-08 12:00:00    207
2012-04-08 11:55:00    207
2012-04-08 11:50:00    207
2012-04-08 11:45:00    207
2012-04-08 11:40:00    207
2012-04-08 11:30:00    207
2012-04-08 12:35:00    207
2012-04-08 11:25:00    207
2012-04-08 11:20:00    207
2012-04-08 11:15:00    207
2012-04-08 11:10:00    207
dtype: int64

In [43]:
# ============================================================
# Completely-zero timestamps
# ============================================================

all_zero_mask = (traffic_df == 0).all(axis=1)
all_zero_timestamps = traffic_df.index[all_zero_mask]

print("Completely-zero timestamps:", len(all_zero_timestamps))

print("\nFirst 30 completely-zero timestamps:")
display(all_zero_timestamps[:30])

Completely-zero timestamps: 2148

First 30 completely-zero timestamps:


DatetimeIndex(['2012-03-01 00:15:00', '2012-03-01 00:20:00',
               '2012-03-03 00:15:00', '2012-03-05 13:30:00',
               '2012-03-05 13:35:00', '2012-03-05 13:40:00',
               '2012-03-05 13:45:00', '2012-03-05 13:50:00',
               '2012-03-05 13:55:00', '2012-03-05 14:00:00',
               '2012-03-05 14:05:00', '2012-03-05 14:10:00',
               '2012-03-05 14:15:00', '2012-03-05 14:20:00',
               '2012-03-05 14:25:00', '2012-03-05 14:30:00',
               '2012-03-05 14:35:00', '2012-03-05 14:40:00',
               '2012-03-05 14:45:00', '2012-03-05 14:50:00',
               '2012-03-06 11:55:00', '2012-03-06 12:00:00',
               '2012-03-06 14:10:00', '2012-03-06 14:15:00',
               '2012-03-06 16:00:00', '2012-03-06 16:05:00',
               '2012-03-06 18:30:00', '2012-03-06 18:35:00',
               '2012-03-06 22:55:00', '2012-03-06 23:00:00'],
              dtype='datetime64[ns]', name='timestamp', freq=None)

In [44]:
# ============================================================
# Consecutive completely-zero periods
# ============================================================

zero_runs = []

if len(all_zero_timestamps) > 0:
    start = all_zero_timestamps[0]
    previous = all_zero_timestamps[0]

    for timestamp in all_zero_timestamps[1:]:
        if timestamp - previous == pd.Timedelta(minutes=5):
            previous = timestamp
        else:
            zero_runs.append((start, previous))
            start = timestamp
            previous = timestamp

    zero_runs.append((start, previous))

zero_runs_df = pd.DataFrame(
    zero_runs,
    columns=["start", "end"]
)

zero_runs_df["duration_minutes"] = (
    (zero_runs_df["end"] - zero_runs_df["start"])
    / pd.Timedelta(minutes=1)
) + 5

zero_runs_df["number_of_timesteps"] = (
    zero_runs_df["duration_minutes"] / 5
).astype(int)

print("Number of completely-zero periods:", len(zero_runs_df))

display(
    zero_runs_df.sort_values(
        "duration_minutes",
        ascending=False
    ).head(20)
)

Number of completely-zero periods: 97


,start,end,duration_minutes,number_of_timesteps
30,2012-03-26 08:25:00,2012-03-27 16:05:00,1905.0,381
90,2012-06-17 21:15:00,2012-06-18 11:00:00,830.0,166
88,2012-06-16 21:30:00,2012-06-17 11:15:00,830.0,166
52,2012-04-25 00:15:00,2012-04-25 13:10:00,780.0,156
36,2012-04-08 04:10:00,2012-04-08 14:40:00,635.0,127
71,2012-06-01 02:55:00,2012-06-01 12:55:00,605.0,121
83,2012-06-09 04:05:00,2012-06-09 13:50:00,590.0,118
16,2012-03-08 19:05:00,2012-03-09 03:55:00,535.0,107
38,2012-04-08 20:15:00,2012-04-09 03:55:00,465.0,93
39,2012-04-09 12:50:00,2012-04-09 18:45:00,360.0,72


In [45]:
# ============================================================
# Zero behaviour for the worst individual sensor
# ============================================================

worst_sensor = zero_stats["zero_percentage"].idxmax()

sensor_zero_mask = traffic_df[worst_sensor] == 0

print("Worst sensor:", worst_sensor)
print(
    "Zero count:",
    int(sensor_zero_mask.sum())
)
print(
    "Zero percentage:",
    zero_stats.loc[worst_sensor, "zero_percentage"]
)

print("\nFirst 30 zero timestamps:")
display(
    traffic_df.index[sensor_zero_mask][:30]
)

Worst sensor: 718076
Zero count: 6890
Zero percentage: 20.10387488328665

First 30 zero timestamps:


DatetimeIndex(['2012-03-01 00:00:00', '2012-03-01 00:05:00',
               '2012-03-01 00:10:00', '2012-03-01 00:15:00',
               '2012-03-01 00:20:00', '2012-03-01 00:25:00',
               '2012-03-01 00:30:00', '2012-03-01 00:35:00',
               '2012-03-01 00:40:00', '2012-03-01 00:45:00',
               '2012-03-01 00:50:00', '2012-03-01 00:55:00',
               '2012-03-01 01:00:00', '2012-03-01 01:05:00',
               '2012-03-01 01:10:00', '2012-03-01 02:45:00',
               '2012-03-01 02:50:00', '2012-03-01 02:55:00',
               '2012-03-01 03:00:00', '2012-03-01 03:05:00',
               '2012-03-01 03:10:00', '2012-03-01 03:15:00',
               '2012-03-01 03:20:00', '2012-03-01 03:25:00',
               '2012-03-01 03:30:00', '2012-03-01 03:50:00',
               '2012-03-01 04:10:00', '2012-03-01 04:15:00',
               '2012-03-01 04:20:00', '2012-03-01 04:25:00'],
              dtype='datetime64[ns]', name='timestamp', freq=None)